# 🏙️ 09 · Build an in silico cell: crowding, membranes and realistic tomograms
### Packing molecules like a cell does, imaging them like a microscope does — and knowing the answer key

**BiologyToResearch** · Part B — Imaging physics · Module 09 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/09_in_silico_cell_simulation.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn |
| **Prerequisites** | [02 · Rotations](02_rotations_geometry.ipynb), [06 · Image formation & CTF](06_electron_image_formation.ipynb), [07 · Tomography & missing wedge](07_tomography_missing_wedge.ipynb) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Pei *et al.* 2016 (crowded-cell simulation, BMC Bioinformatics); Liu *et al.* 2020 (unified packing of deformable and rigid structures, BMC Bioinformatics); Liu *et al.* 2020 (efficient crowding simulation, SARS-CoV-2, BIBM); Gubins *et al.* SHREC'19 / SHREC'20 benchmarks; Wang *et al.* 2018 (pseudo-structures, BMVC) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments in this notebook are small, controlled *toy versions* of published methods. They illustrate the core
> idea of each paper; they do **not** reproduce the papers' numbers. **All molecules and data are generated by code**
> (nothing is downloaded): realistic stand-ins with the size and symmetry of the PDB entries they are named after.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why simulated tomograms are indispensable (ground truth, benchmarks, training data) and what they cannot do.
2. **Implement** packing of molecules as bounding spheres by **rejection sampling** and by **simulated annealing**
   (overlap energy + Metropolis rule + temperature schedule), and **measure** which crowding levels each can reach.
3. **Describe** how the lab's later simulators represent molecules as *several* balls and membranes/filaments as bonded,
   bendable chains (Liu *et al.* 2020), and how a target can be packed with close neighbours (Liu *et al.* BIBM 2020).
4. **Run** a full physical simulation of a small crowded volume — random rotations → tilt series → CTF and noise per tilt
   → weighted back-projection — and **compare** it with the course's fast simulator.
5. **Measure** how crowding changes the "SNR" of a tomogram and the contamination of a particle's box by neighbours.
6. **Save** a ground-truth table and **relate** it to community benchmarks (SHREC) and to pseudo-structures for
   statistical testing (Wang *et al.* 2018).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, load tools and molecules | 5 min |
| 1 · Why simulate? | what a simulator buys you | 5 min |
| 2 · Bounding spheres and crowding | turn molecules into spheres, define volume fraction | 10 min |
| 3 · Rejection sampling | random placement and its "jamming" limit | 10 min |
| 4 · Simulated-annealing packing | overlap energy, Metropolis, cooling — Pei *et al.* 2016 | 20 min |
| 5 · Beyond one sphere | multi-ball molecules, bendable membranes, target + neighbours — Liu *et al.* 2020 ×2 | 20 min |
| 6 · From spheres to densities | render the cells; crowding vs SNR and neighbour contamination | 10 min |
| 7 · Physical vs fast simulation | tilt series + CTF + noise + WBP vs the toolkit shortcut | 20 min |
| 8 · Answer key and benchmarks | ground-truth table, SHREC | 10 min |
| 9 · Pseudo-structures and the limits of simulation | Wang *et al.* 2018; domain shift | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

The cytoplasm of *E. coli* contains roughly 300–400 grams of macromolecules per litre: about a fifth to a third of its
volume is occupied by ribosomes, chaperones, proteasome-like machines, metabolic enzymes and nucleic acids. This
**macromolecular crowding** changes how molecules diffuse, fold and bind — and it is exactly what makes cellular
tomograms so hard to analyse: neighbours touch, overlap in projection, and contaminate every particle's box.

Suppose you invent a new particle picker or classifier. How do you know it works? In a real cell nobody knows where
every ribosome is. The answer of the Xu Lab (and the whole field) is **simulation**: build an *in silico* cell whose
composition, positions and orientations you control, image it with a virtual microscope, and score your method against
the known answer. A simulator is only useful if it is **realistic** in the ways that matter — crowding, membranes,
CTF, missing wedge, noise. This notebook builds one, piece by piece.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "(not needed here)")
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import csv
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from sklearn.cluster import KMeans

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook uses from [`course_toolkit.py`](../course_toolkit.py). New today:
`support_radius` (a molecule's bounding sphere), `ctf_2d` (Module 06), `project_tilt_series` and
`weighted_back_projection` (Module 07) and the fast simulator `simulate_tomogram`, which we will finally put to the test.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def rot_y(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def euler_zyz_to_matrix(phi, theta, psi):
    """R = Rz(phi) · Ry(theta) · Rz(psi)  (radians, acting on (x, y, z) vectors)."""
    return rot_z(phi) @ rot_y(theta) @ rot_z(psi)


def matrix_to_euler_zyz(R):
    """Inverse of :func:`euler_zyz_to_matrix` (one of the two equivalent solutions)."""
    theta = np.arccos(np.clip(R[2, 2], -1, 1))
    if np.isclose(np.sin(theta), 0):
        return np.arctan2(R[1, 0], R[0, 0]), theta, 0.0
    return np.arctan2(R[1, 2], R[0, 2]), theta, np.arctan2(R[2, 1], -R[2, 0])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def project_tilt_series(vol: np.ndarray, angles_deg) -> np.ndarray:
    """Tilt-series projections of vol[z, y, x] about the y axis → array[n_tilts, y, x]."""
    return np.stack([ndimage.rotate(vol, a, axes=(0, 2), reshape=False, order=1).sum(axis=0)
                     for a in angles_deg]).astype(np.float32)


def weighted_back_projection(tilts: np.ndarray, angles_deg, nz: int) -> np.ndarray:
    """Weighted (ramp-filtered) back-projection of a y-axis tilt series → vol[nz, y, x]."""
    n, ny, nx = tilts.shape
    ramp = np.abs(np.fft.fftfreq(nx))[None, None, :]
    filt = np.real(np.fft.ifft(np.fft.fft(tilts, axis=2) * ramp, axis=2))
    rec = np.zeros((nz, ny, nx), np.float32)
    for img, a in zip(filt, angles_deg):
        smear = np.broadcast_to(img[None], (nz, ny, nx))
        rec += ndimage.rotate(smear, -a, axes=(0, 2), reshape=False, order=1)
    return rec / n


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0


def simulate_tomogram(clean: np.ndarray, rng, snr: float = 0.05, tilt_max_deg: float = 60.0,
                      ctf_pixel_A: float | None = 10.0, defocus_um: float = 5.0) -> np.ndarray:
    """Fast tomogram simulation: (optional 3D CTF-like filter) → missing wedge → noise.

    The CTF is applied isotropically here (a simplification of per-tilt CTF).
    """
    v = clean
    if ctf_pixel_A:
        ctf = ctf_2d(clean.shape, ctf_pixel_A, defocus_um=defocus_um, kV=300, B=0)
        v = apply_fourier_mask(v, -ctf)          # sign flip: dense objects appear bright
    v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng)
print("🧰 toolkit loaded")

In [ ]:
# ── Data: the four star molecules of the course ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]          # ribosome, 20S proteasome, capped proteasome, RuBisCO
zoo = load_zoo(CAST, size=32, sigma=1.0)
NAMES = {pid: ZOO[pid]["name"] for pid in CAST}

---
> 📍 **Where we are.** Module 06 showed how one projection image forms (projection, CTF, dose, noise); Module 07 turned
> a tilt series into a tomogram and met the missing wedge. Both worked with a handful of molecules placed by hand. Real
> cells are *crowded*, and every method in Parts C–F is benchmarked on simulated crowded tomograms. This module builds
> such a simulator — and checks the fast shortcut (`simulate_tomogram`) that the rest of the course relies on.

## 1 · Why simulate?

| What you need | Real tomograms | Simulated tomograms |
|---|---|---|
| **Ground truth** (identity, position, orientation of every molecule) | unknown (only expert guesses) | known exactly |
| **Control** over one factor at a time (crowding, SNR, tilt range, defocus) | impossible | trivial |
| **Unlimited training data** for deep networks (Modules 16–20) | expensive to label | free |
| **Realism** (noise statistics, ice, damage, unknown molecules) | perfect | approximate → *domain shift* (Module 20) |

A simulator has two halves: **(1) building the specimen** — which molecules, how many, where, in which orientation,
without overlapping; and **(2) imaging it** — tilt series, CTF, noise, reconstruction. Sections 2–6 build the specimen,
Section 7 images it.

---
## 2 · Bounding spheres and crowding

Checking whether two irregular molecules overlap is expensive. The classic trick (used by Pei *et al.* 2016) is to
replace each molecule by its **bounding sphere**: the smallest sphere that contains the molecule's density above a
contour level (Pei *et al.* used 0.2 × the maximum density). Two molecules cannot overlap if their spheres do not:

$$d_{ij}\;=\;\lVert\mathbf c_i-\mathbf c_j\rVert\;\ge\;r_i+r_j .$$

*In words:* the distance between the two centres must be at least the sum of the two radii.

The crowding level is measured as the **volume fraction** $\phi$ — the fraction of the box occupied by the spheres:

$$\phi=\frac{\sum_i \tfrac43\pi r_i^3}{V_{\text{box}}}.$$

Pei *et al.* simulated 11 %, 26 % and 44 % (2,000, 5,000 or 8,000 complexes in a 500 × 500 × 200 nm box).

In [ ]:
# ── 2.1 Bounding spheres of the cast (contour at 0.2 × max, as in Pei et al. 2016) ──
RAD = {pid: support_radius(zoo[pid], level=0.2) for pid in CAST}
fig, axes = plt.subplots(1, 4, figsize=(9.6, 2.6))
for ax, pid in zip(axes, CAST):
    ax.imshow(zoo[pid].sum(axis=1), extent=(-16, 16, 16, -16))
    ax.add_patch(plt.Circle((0, 0), RAD[pid], fill=False, color="tab:orange", lw=1.5))
    ax.set_title(f"{pid.upper()} {NAMES[pid]}\nr = {RAD[pid]:.1f} nm", fontsize=8)
    ax.set_xlim(-17, 17); ax.set_ylim(17, -17); ax.set_xlabel("x (nm)")
axes[0].set_ylabel("z (nm)")
plt.tight_layout(); plt.show()
for pid in CAST:
    v_sphere = 4 / 3 * np.pi * RAD[pid] ** 3
    v_mol = (zoo[pid] > 0.2 * zoo[pid].max()).sum()
    print(f"{NAMES[pid]:26s} sphere {v_sphere:7.0f} nm³ vs molecule {v_mol:5d} nm³ → sphere wastes {100 * (1 - v_mol / v_sphere):.0f} %")

🔍 **What just happened?** Each molecule got an orange bounding sphere (our `support_radius` uses the smallest sphere
centred on the molecule's centre of mass — a slight overestimate of the true minimum bounding sphere). Spheres waste a
lot of space, especially for elongated molecules such as the capped proteasome; keep that in mind for Section 5.

A recipe for a cell: we choose a **composition** (number fractions of each molecule) and keep adding molecules until
their spheres reach the target volume fraction. (Pei *et al.* drew copy numbers from a multinomial distribution with
random frequencies over 21 complexes; we use fixed fractions of our four.)

In [ ]:
# ── 2.2 Choose which molecules go into a cell of a given crowding level ──
MIX = {"4v4r": 0.25, "3j9i": 0.25, "6utj": 0.15, "1bxn": 0.35}       # number fractions (sum to 1)


def draw_composition(box, target_fraction: float, rng) -> tuple[list, np.ndarray]:
    """Draw molecule classes at random (fractions MIX) until their spheres fill ``target_fraction`` of the box."""
    classes, radii, filled = [], [], 0.0
    while filled < target_fraction * np.prod(box):
        c = str(rng.choice(list(MIX), p=list(MIX.values())))
        classes.append(c); radii.append(RAD[c])
        filled += 4 / 3 * np.pi * RAD[c] ** 3
    return classes, np.array(radii)


def volume_fraction(radii, box) -> float:
    """Sum of sphere volumes / box volume."""
    return float((4 / 3 * np.pi * np.asarray(radii) ** 3).sum() / np.prod(box))


BOX = (64, 128, 128)                                                # z, y, x in nm (1 voxel = 1 nm)
TARGETS = (0.11, 0.26, 0.44)                                        # Pei et al.'s three crowding levels
compositions = {phi: draw_composition(BOX, phi, np.random.default_rng(1)) for phi in TARGETS}
for phi, (cls, rad) in compositions.items():
    counts = {NAMES[p]: cls.count(p) for p in CAST}
    print(f"target {100 * phi:.0f} %: {len(cls):3d} molecules {counts}")

---
## 3 · Rejection sampling: the simplest packer — and where it gets stuck

**Rejection sampling** (also called *random sequential addition*): take the molecules one at a time; propose a random
centre inside the box; if its sphere overlaps an already-placed sphere (or sticks out of the box), reject and try again;
give up after a fixed number of tries. The course toolkit's `make_crowded_cell` (used in Module 00) works this way.

💡 Physics tells us this cannot go far: for equal spheres placed at random and never moved again, the box "jams" at about
38 % volume fraction in 3D (about 55 % for disks in 2D) — later spheres find no hole big enough. With finite tries, box
walls and big ribosomes, the practical limit is lower.

In [ ]:
# ── 3.1 Rejection sampling (random sequential addition) ──
def rejection_pack(radii: np.ndarray, box, rng, tries: int = 500):
    """Place spheres one by one at random non-overlapping positions; returns (centres, indices of placed spheres)."""
    box = np.asarray(box, float)
    centres, placed = [], []
    for i in rng.permutation(len(radii)):
        r = radii[i]
        for _ in range(tries):
            c = rng.uniform(r, box - r)
            if not centres or np.all(np.linalg.norm(np.array(centres) - c, axis=1) >= radii[placed] + r):
                centres.append(c); placed.append(i)
                break
    return np.array(centres), np.array(placed)


t0 = time.time()
rsa_results = {}
for phi, (cls, rad) in compositions.items():
    cen, placed = rejection_pack(rad, BOX, np.random.default_rng(2))
    rsa_results[phi] = (cen, placed)
    dropped = [cls[i] for i in sorted(set(range(len(cls))) - set(placed.tolist()))]
    print(f"target {100 * phi:.0f} %: placed {len(placed)}/{len(rad)} molecules, "
          f"reached {100 * volume_fraction(rad[placed], BOX):.1f} %; dropped: "
          + (", ".join(f"{dropped.count(p)} × {NAMES[p]}" for p in CAST if dropped.count(p)) or "none"))
print(f"({time.time() - t0:.1f} s)")

🔍 **What just happened?** At 11 % every molecule finds a place; at 26 % some already fail; at 44 % rejection sampling
stalls far below the target — about a quarter of the molecules are simply dropped, and the dropped ones are the *big*
ones (ribosomes and capped proteasomes need the biggest holes), which biases the composition of the simulated cell.
To reach cell-like crowding we must let molecules **move** after being placed.

---
## 4 · Packing by simulated annealing

**Idea.** Start with all molecules at random positions, overlaps allowed. Define an **energy** that is zero only when
nothing overlaps, and then *shake* the system while slowly "cooling" it, like annealing metal so its atoms settle into a
low-energy arrangement.

**Energy (overlap penalty).**

$$E=\sum_{i<j}\big[\max(0,\;r_i+r_j-d_{ij})\big]^2\;+\;\sum_i\sum_{a\in\{x,y,z\}}\Big(\big[\max(0,\,r_i-c_{i,a})\big]^2+\big[\max(0,\,c_{i,a}+r_i-L_a)\big]^2\Big).$$

*In words:* every pair of spheres that overlaps adds the square of the overlap depth (in nm²); every sphere sticking out
of the box (side lengths $L_a$) adds the square of how far it sticks out. $E=0$ ⇔ a valid packing.

**Moves and the Metropolis rule.** Pick a random sphere, propose a small random displacement, and compute the energy
change $\Delta E$. Accept the move with probability

$$P_{\text{acc}}=\min\!\big(1,\;e^{-\Delta E/T}\big).$$

*In words:* moves that reduce overlap are always accepted; moves that *increase* it are sometimes accepted — often when
the "temperature" $T$ is high, almost never when it is low. Accepting bad moves early lets the system escape traffic jams
(local minima).

**Temperature schedule.** Geometric cooling, $T_k=T_0\,\alpha^k$ after sweep $k$ (one sweep = one proposed move per
sphere), with $\alpha=(T_{\text{end}}/T_0)^{1/K}$ chosen so that $T$ falls from $T_0$ to $T_{\text{end}}$ in $K$ sweeps;
we finish with a few sweeps at $T=0$ (pure downhill, a "quench"). The step size adapts so that about 40 % of moves are
accepted.

In [ ]:
# ── 4.1 Simulated-annealing packer ──
def wall_energy(c, r: float, box) -> float:
    """Squared distance by which a sphere of radius r at centre c sticks out of the box."""
    e = 0.0
    for a in range(3):
        lo, hi = r - c[a], c[a] + r - box[a]
        e += lo * lo if lo > 0 else 0.0
        e += hi * hi if hi > 0 else 0.0
    return e


def delta_energy(i: int, new, centres: np.ndarray, radii: np.ndarray, box) -> float:
    """Energy change when sphere i moves from centres[i] to new (pair overlaps + walls)."""
    P = np.stack((centres[i], new))
    d = np.sqrt(((centres[None, :, :] - P[:, None, :]) ** 2).sum(-1))       # (2, N): old and new distances
    ov = radii[None, :] + radii[i] - d
    ov[:, i] = 0.0
    np.maximum(ov, 0.0, out=ov)
    e_pair = (ov * ov).sum(1)
    return (e_pair[1] + wall_energy(new, radii[i], box)) - (e_pair[0] + wall_energy(centres[i], radii[i], box))


def total_energy(centres: np.ndarray, radii: np.ndarray, box) -> float:
    """Overlap energy E of a whole packing (nm²)."""
    d = np.linalg.norm(centres[:, None] - centres[None], axis=-1)
    ov = np.clip(radii[:, None] + radii[None] - d, 0, None)
    np.fill_diagonal(ov, 0)
    box = np.asarray(box, float)
    lo = np.clip(radii[:, None] - centres, 0, None); hi = np.clip(centres + radii[:, None] - box, 0, None)
    return float((ov ** 2).sum() / 2 + (lo ** 2).sum() + (hi ** 2).sum())


def max_overlap(centres: np.ndarray, radii: np.ndarray) -> float:
    """Largest pairwise overlap depth r_i + r_j − d_ij (nm); ≤ 0 means no overlap at all."""
    d = np.linalg.norm(centres[:, None] - centres[None], axis=-1)
    ov = radii[:, None] + radii[None] - d
    np.fill_diagonal(ov, -np.inf)
    return float(ov.max())


def sa_pack(radii: np.ndarray, box, rng, n_sweeps: int = 300, T0: float = 10.0, T_end: float = 1e-3,
            step0: float = 3.0, quench: int = 50):
    """Simulated-annealing packing of spheres in a box. Returns (centres, history of (T, E, acceptance rate))."""
    box = [float(b) for b in box]
    n = len(radii)
    centres = rng.uniform(0, 1, (n, 3)) * np.array(box)                   # random start, overlaps allowed
    alpha = (T_end / T0) ** (1 / n_sweeps)
    T, step, history = T0, step0, []
    for sweep in range(n_sweeps + quench):
        Tk = T if sweep < n_sweeps else 0.0
        who, moves, coins = rng.integers(n, size=n), rng.normal(0, step, (n, 3)), rng.random(n)
        accepted = 0
        for i, mv, u in zip(who, moves, coins):
            new = centres[i] + mv
            dE = delta_energy(i, new, centres, radii, box)
            if dE <= 0 or (Tk > 0 and u < math.exp(-dE / Tk)):           # the Metropolis rule
                centres[i] = new
                accepted += 1
        rate = accepted / n
        step = float(np.clip(step * (1.1 if rate > 0.4 else 0.9), 0.05, 10.0))
        history.append((Tk, total_energy(centres, radii, box), rate))
        T *= alpha
    return centres, history

In [ ]:
# ── 4.2 Pack the three crowding levels by simulated annealing ──
t0 = time.time()
sa_results = {}
for phi, (cls, rad) in compositions.items():
    cen, hist = sa_pack(rad, BOX, np.random.default_rng(3))
    sa_results[phi] = (cen, hist)
    print(f"target {100 * phi:.0f} %: {len(rad)} spheres, final energy {hist[-1][1]:.3f} nm², "
          f"largest overlap {max_overlap(cen, rad):+.2f} nm")
print(f"annealing took {time.time() - t0:.1f} s")
assert all(max_overlap(c, compositions[p][1]) < 1.0 for p, (c, _) in sa_results.items()), \
    "annealing should leave overlaps below one voxel (1 nm)"

In [ ]:
# ── 4.3 Watch the annealing: temperature, energy, acceptance ──
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))
for phi, (cen, hist) in sa_results.items():
    h = np.array(hist)
    axes[1].semilogy(np.maximum(h[:, 1], 1e-3), label=f"{100 * phi:.0f} %")
    axes[2].plot(h[:, 2], label=f"{100 * phi:.0f} %")
axes[0].semilogy(np.maximum(np.array(sa_results[0.44][1])[:, 0], 1e-4), color="tab:red")
axes[0].set_title("temperature schedule T_k"); axes[0].set_xlabel("sweep"); axes[0].set_ylabel("T (nm²)")
axes[1].set_title("overlap energy E (floored at 10⁻³)"); axes[1].set_xlabel("sweep"); axes[1].set_ylabel("E (nm²)")
axes[2].set_title("acceptance rate"); axes[2].set_xlabel("sweep"); axes[2].set_ylim(0, 1)
for ax in axes[1:]:
    ax.axvline(300, color="gray", ls=":", lw=1)
axes[1].legend(fontsize=7, title="crowding", title_fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At high temperature the energy stays high (the system explores, accepting many bad moves);
as $T$ falls, overlaps are squeezed out; the final quench (after the dotted line, $T=0$) removes the last ones. The 44 %
cell is the hardest: its energy falls latest and its final overlaps, if any, are a fraction of a nanometre (smaller than
one voxel). Now the comparison that matters:

In [ ]:
# ── 4.4 Rejection sampling vs simulated annealing ──
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw=dict(width_ratios=[1.1, 1, 1]))
x = np.arange(len(TARGETS))
reached_rsa = [100 * volume_fraction(compositions[p][1][rsa_results[p][1]], BOX) for p in TARGETS]
reached_sa = [100 * volume_fraction(compositions[p][1], BOX) for p in TARGETS]
axes[0].bar(x - 0.2, reached_rsa, 0.4, label="rejection sampling")
axes[0].bar(x + 0.2, reached_sa, 0.4, label="simulated annealing")
axes[0].plot(x, [100 * p for p in TARGETS], "k_", ms=30, mew=2, label="target")
axes[0].set_xticks(x, [f"{100 * p:.0f} %" for p in TARGETS]); axes[0].set_xlabel("target crowding")
axes[0].set_ylabel("volume fraction reached (%)"); axes[0].legend(fontsize=7)


def draw_slice(ax, centres, radii, classes, z0, title):
    """Draw the circles where the spheres cut the plane z = z0."""
    colors = {"4v4r": "tab:red", "3j9i": "tab:blue", "6utj": "tab:cyan", "1bxn": "tab:green"}
    for c, r, k in zip(centres, radii, classes):
        dz = abs(c[0] - z0)
        if dz < r:
            ax.add_patch(plt.Circle((c[2], c[1]), np.sqrt(r * r - dz * dz), color=colors[k], alpha=0.6))
    ax.set_xlim(0, BOX[2]); ax.set_ylim(0, BOX[1]); ax.set_aspect("equal")
    ax.set_title(title, fontsize=9); ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")


cls44, rad44 = compositions[0.44]
cen_r, placed_r = rsa_results[0.44]
draw_slice(axes[1], cen_r, rad44[placed_r], [cls44[i] for i in placed_r], BOX[0] / 2, "44 %: rejection sampling")
draw_slice(axes[2], sa_results[0.44][0], rad44, cls44, BOX[0] / 2, "44 %: simulated annealing")
plt.tight_layout(); plt.show()
for p, a, b in zip(TARGETS, reached_rsa, reached_sa):
    print(f"target {100 * p:.0f} %: rejection {a:.1f} %  |  annealing {b:.1f} %")

🔍 **What just happened?** Simulated annealing reaches every target, including the 44 % that rejection sampling cannot
approach; the slices (circles = where spheres cut the mid-plane; red ribosomes, blue 20S, cyan capped 20S, green RuBisCO)
show the denser, more cell-like packing. The price is compute: every molecule moves hundreds of times.

✏️ **Your turn.** In 4.2, try `sa_pack(rad, BOX, rng, n_sweeps=60)` (cooling 5× faster) for the 44 % cell. Does the final
energy still reach ~0? What happens with `T0=0.01` (almost no annealing — a pure "downhill" packer)?

### 📄 Paper connection — Pei *et al.* 2016: simulating crowded cytoplasm to test particle picking

**Paper**: Pei L, Xu M, Frazier Z, Alber F. *Simulating cryo electron tomograms of crowded cell cytoplasm for assessment
of automated particle picking.* BMC Bioinformatics 17:405 (2016). [doi:10.1186/s12859-016-1283-3](https://doi.org/10.1186/s12859-016-1283-3)
(⚠️ title note: some lists, including AITom's publication list, paraphrase it as "…Crowded Mixtures of Macromolecular
Complexes and Assessment of Particle Picking"; the published title is the one above.)

- **Question it answers:** how well does reference-free particle picking work in realistically **crowded** cytoplasm,
  where the ground truth of real data is unknown?
- **What the paper actually does:**
  1. *Packing* — 21 abundant complexes converted to density maps (PDB2VOL, 1 nm voxels) and represented by minimum
     bounding spheres (contour at 0.2 × maximum density); copy numbers drawn from a multinomial with random frequencies;
     spheres packed without overlap using IMP **molecular dynamics plus simulated annealing** (annealing parameters
     reported as $T_0=3000$, $c=100$), followed by conjugate-gradient minimisation.
  2. *Imaging* — random orientations; tilts −60°…+60° in 2° steps; 300 kV, $C_s$ = 2 mm, defocus −4 µm (their sign
     convention for underfocus); CTF and detector MTF $\mathrm{sinc}(\pi\omega/2)$; noise added before and after the CTF;
     SNR ∈ {50, 20, 10, 1}; back-projection.
  3. *Picking* — difference-of-Gaussians (DoG) peaks scored against the known positions (Module 11).
- **What our toy keeps:** bounding spheres, crowding levels 11/26/44 %, an overlap energy minimised by simulated
  annealing, and (Section 7) the same tilt scheme and imaging physics.
- **What our toy leaves out:** molecular dynamics (we use Monte-Carlo moves), IMP's force field and schedule, 21
  complexes (we use 4), the 500 × 500 × 200 nm box, noise before the CTF, and the picking study itself.
- **Result reported:** with crowding rising from 11 % to 26 % to 44 %, the authors report that maximum DoG recall falls
  from 78.7 % to 63.2 % to 52.0 % (σ = 7 nm), and the best F-score is 0.831 at 11 % crowding and SNR 50.
- **Code:** AITom `aitom/simulation/tomogram/single_bounding_sphere/` (packing + simulation) and `aitom/pick/dog/`.

---
## 5 · Beyond one sphere: multi-ball molecules, bendable membranes, target + neighbours

### 5.1 Several balls fit better than one

A single sphere around a capped proteasome is mostly empty space. Liu *et al.* (2020, BMC Bioinformatics) represent
each macromolecule by **several balls**: run **k-means** on its atom coordinates, and give each ball the radius of the
farthest atom in its cluster; the balls are bonded rigidly together. (They used $k=\max(3,\lceil N_{\text{atom}}/5000\rceil)$
balls.) We have voxels instead of atoms, so we cluster the coordinates of the voxels above the 0.2 contour.

In [ ]:
# ── 5.1 Multi-ball representation by k-means ──
def multi_ball(template: np.ndarray, k: int, level: float = 0.2, seed: int = 0):
    """Represent a molecule by k balls: k-means on its voxel coordinates; radius = farthest member voxel (+0.5)."""
    pts = np.argwhere(template > level * template.max()).astype(float)
    if k == 1:
        centres, labels = pts.mean(0, keepdims=True), np.zeros(len(pts), int)
    else:
        km = KMeans(n_clusters=k, n_init=4, random_state=seed).fit(pts)
        centres, labels = km.cluster_centers_, km.labels_
    radii = np.array([np.linalg.norm(pts[labels == j] - centres[j], axis=1).max() + 0.5 for j in range(k)])
    return centres, radii


def union_volume(centres, radii, size: int = 32) -> int:
    """Number of voxels (nm³) inside at least one ball."""
    g = np.stack(np.meshgrid(*[np.arange(size)] * 3, indexing="ij"), -1).reshape(-1, 3)
    inside = np.zeros(len(g), bool)
    for c, r in zip(centres, radii):
        inside |= np.linalg.norm(g - c, axis=1) <= r
    return int(inside.sum())


K_LIST = [1, 3, 24]
fig, axes = plt.subplots(2, 3, figsize=(8.4, 5.4))
for row, pid in enumerate(["6utj", "4v4r"]):
    for col, k in enumerate(K_LIST):
        cen_b, rad_b = multi_ball(zoo[pid], k)
        ax = axes[row, col]
        ax.imshow(zoo[pid].sum(axis=1))                                # view down y: image rows = z, columns = x
        for c, r in zip(cen_b, rad_b):
            ax.add_patch(plt.Circle((c[2], c[0]), r, fill=False, color="tab:orange", lw=0.9))
        ax.set_title(f"{NAMES[pid]}, k = {k}\nballs: {union_volume(cen_b, rad_b)} nm³", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
for pid in CAST:
    vols = {k: union_volume(*multi_ball(zoo[pid], k)) for k in K_LIST}
    mol = int((zoo[pid] > 0.2 * zoo[pid].max()).sum())
    print(f"{NAMES[pid]:26s} molecule {mol:5d} nm³ | 1 ball {vols[1]:6d} nm³ | "
          + " | ".join(f"{k} balls {vols[k]:6d} nm³ ({100 * (vols[k] / vols[1] - 1):+.0f} %)" for k in K_LIST[1:]))

🔍 **What just happened?** For the elongated capped proteasome, even three balls hug the shape and need far less volume
than one sphere — that is what lets multi-ball packers reach realistic crowding without artificial gaps. For compact
shapes such as the 20S barrel and RuBisCO, three balls do *not* help (they can even need more volume: each ball must
reach the farthest voxel of its cluster), and the ribosome gains only a little; many small balls help every molecule.
That is why the paper's rule $k=\max(3,\lceil N_{\text{atom}}/5000\rceil)$ gives big complexes many balls (a 70S
ribosome has roughly 1.5 × 10⁵ non-hydrogen atoms → about 30 balls). (Circles are drawn in
projection.) Liu *et al.* report savings of 11–72 % for 75 % of their macromolecules.

### 5.2 Bendable structures: a membrane as a chain of bonded beads

Membranes and filaments cannot be bounding spheres at all: they are long and they **bend**. In Liu *et al.*'s unified
framework, a filament is a chain of balls and a membrane a 2D mesh of balls; neighbouring balls are joined by springs
(bond energy $U^{bond}=k_b(\lVert x_{ij}\rVert-\lVert x_0\rVert)^2$), and the angle-bending constant is set to
$k_\theta=0$ so they **bend but do not stretch**. Everything repels everything (Lennard-Jones in the paper), and a
**central compression force** pushes all objects together; the system is then integrated with molecular dynamics.

Our 2D toy keeps the spirit: a membrane = a chain of beads with bond springs (no bending penalty), anchored at both ends;
molecules = disks pulled toward the centre; a soft repulsion $U=k_r(\sigma_{ij}-d_{ij})^2$ for $d_{ij}<\sigma_{ij}$; and
simple *overdamped* dynamics (each step moves every bead a little along its total force).

In [ ]:
# ── 5.2 A 2D bead-spring toy: rigid molecules pressing on a bendable membrane ──
def bead_spring_2d(n_mol: int = 24, n_beads: int = 46, n_steps: int = 4000, seed: int = 0):
    """Overdamped 2D packing of disks (molecules) around an anchored bead chain (membrane). Returns (start, end, radii, n_mol)."""
    r_ng = np.random.default_rng(seed)
    r_mol = r_ng.choice([4.0, 5.5, 7.0], n_mol)                       # disk radii (nm)
    xs = np.linspace(-55, 55, n_beads)
    chain = np.stack([xs, 8 * np.sin(xs / 55 * 2 * np.pi)], 1)        # a wavy membrane, longer than its end-to-end span
    X = np.vstack([r_ng.uniform(-50, 50, (n_mol, 2)), chain])
    R = np.concatenate([r_mol, np.full(n_beads, 1.5)])
    N = len(X)
    bonds = np.array([(n_mol + i, n_mol + i + 1) for i in range(n_beads - 1)])
    bonded = np.zeros((N, N), bool); bonded[bonds[:, 0], bonds[:, 1]] = True; bonded |= bonded.T
    pinned = np.zeros(N, bool); pinned[[n_mol, N - 1]] = True          # membrane ends anchored
    is_mol = np.arange(N) < n_mol
    k_b, k_r, kappa, eta, b0 = 5.0, 5.0, 0.02, 0.05, 3.0
    X0 = X.copy()
    for _ in range(n_steps):
        diff = X[:, None] - X[None]
        d = np.linalg.norm(diff, axis=-1) + np.eye(N)
        sig = R[:, None] + R[None]
        ov = np.where((d < sig) & ~bonded & ~np.eye(N, dtype=bool), sig - d, 0.0)
        F = ((2 * k_r * ov / d)[..., None] * diff).sum(1)               # soft repulsion
        bd = X[bonds[:, 1]] - X[bonds[:, 0]]
        bl = np.linalg.norm(bd, axis=1)
        fb = (2 * k_b * (bl - b0) / bl)[:, None] * bd                   # bond springs (no angle term: free bending)
        np.add.at(F, bonds[:, 0], fb); np.add.at(F, bonds[:, 1], -fb)
        F[is_mol] += -kappa * X[is_mol]                                 # central compression on molecules
        F[pinned] = 0.0
        X = X + eta * F
    return X0, X, R, n_mol


t0 = time.time()
X_start, X_end, R_beads, n_mol2d = bead_spring_2d()
print(f"4000 steps in {time.time() - t0:.1f} s")
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
for ax, Xs, t in zip(axes, [X_start, X_end], ["start", "after compression"]):
    for p, r in zip(Xs[:n_mol2d], R_beads[:n_mol2d]):
        ax.add_patch(plt.Circle(p, r, color="tab:blue", alpha=0.6))
    ax.plot(Xs[n_mol2d:, 0], Xs[n_mol2d:, 1], "-o", ms=2.5, color="tab:red", label="membrane (bead chain)")
    ax.set_aspect("equal"); ax.set_xlim(-62, 62); ax.set_ylim(-62, 62); ax.set_title(t)
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
axes[0].legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** As the molecules (blue) are pushed together, the membrane (red) neither breaks nor stretches —
its bonds keep their length — but it **bends** around the crowd on both sides. Rigid molecules and deformable membranes
live in one simulation with one set of rules: that is the "unified" idea.

### 📄 Paper connection — Liu *et al.* 2020 (BMC Bioinformatics): a unified packing framework

**Paper**: Liu S, Ban X, Zeng X, Zhao F, Gao Y, Wu W, Zhang H, Chen F, Hall T, Gao X, Xu M. *A unified framework for
packing deformable and non-deformable subcellular structures in crowded cryo-electron tomogram simulation.* BMC
Bioinformatics 21:399 (2020). [doi:10.1186/s12859-020-03660-w](https://doi.org/10.1186/s12859-020-03660-w)

- **Question it answers:** how to pack rigid macromolecules *and* deformable membranes and filaments together, more
  tightly and realistically than single bounding spheres allow.
- **What the paper actually does:** everything is bonded balls — macromolecules = $k$ balls from k-means on atoms
  (fully bonded, rigid), filaments = chains, membranes = 2D meshes with 8-neighbour bonds; energies
  $U^{bond}=k_b(\lVert x_{ij}\rVert-\lVert x_0\rVert)^2$, $U^{angle}=k_\theta(\theta_{ijk}-\theta_0)^2$ and Lennard-Jones
  $U^{LJ}=\varepsilon[(R_{min}/d)^{12}-2(R_{min}/d)^6]$, with $k_b=2000$ and $k_\theta=0$ for deformable structures;
  molecular dynamics in NAMD ($a_i=(F^b_i+F^{nb}_i+F^{ext}_i)/m$, $m=5000$, $\delta t=1$ ns) with a central compression
  force; orientations read back from two bonded vectors as ZYZ Euler angles; then density maps and simulated tomograms.
- **What our toy keeps:** k-means multi-ball molecules; bead chains with bonds but no bending penalty; repulsion; central
  compression.
- **What our toy leaves out:** 3D meshes, Lennard-Jones and real MD (we use soft repulsion and overdamped steps), the 20
  complexes × 20 copies in a 120 nm cube, and the downstream tests (DoG picking, FCN segmentation).
- **Result reported:** the authors report that the multi-ball representation saves 11–72 % of volume for 75 % of
  macromolecules relative to single spheres, and a pixel accuracy of 0.7015 for EDSSN3D segmentation on their simulated
  data.
- **Code:** not linked from the paper; AITom's `aitom/geometry/pack/sphere/few/` implements related single- and
  multi-ball packing.

### 5.3 A target and its close neighbours (fast, for training data)

Training a classifier needs *thousands* of subtomograms, each a target molecule with realistic neighbours around it —
simulating whole cells for that is wasteful. The AITom code accompanying Liu *et al.* (BIBM 2020) packs **one target plus
a few random neighbours** tightly: start from random non-overlapping positions, then repeatedly move each sphere a small,
fixed-length step down the gradient of

$$L=\tfrac12\sum_{i<j}\lVert\mathbf x_i-\mathbf x_j\rVert^2,\qquad \nabla_{\mathbf x_j}L=N\mathbf x_j-\sum_i\mathbf x_i ,$$

*in words:* every sphere steps toward the centre of the group; a step is undone if it would create an overlap; several
random starts are run and the most compact packing is kept.

In [ ]:
# ── 5.3 Gradient compaction of one target + six neighbours ──
def compact_pack(radii: np.ndarray, rng, n_rounds: int = 400, step: float = 0.5, spread: float = 80.0):
    """AITom-style single-ball compaction: normalised gradient steps of L = ½Σ|x_i − x_j|², rejected if they overlap."""
    n, centres = len(radii), []
    for r in radii:                                                     # random non-overlapping start
        while True:
            c = rng.uniform(-spread / 2, spread / 2, 3)
            if all(np.linalg.norm(c - o) >= r + ro for o, ro in zip(centres, radii)):
                centres.append(c); break
    x = np.array(centres)
    for _ in range(n_rounds):
        moved = 0
        for j in rng.permutation(n):
            g = n * x[j] - x.sum(0)
            if np.linalg.norm(g) < 1e-9:
                continue
            new = x[j] - step * g / np.linalg.norm(g)
            d = np.linalg.norm(x - new, axis=1); d[j] = np.inf
            if np.all(d >= radii + radii[j]):
                x[j] = new; moved += 1
        if moved == 0:
            break
    L = 0.5 * sum(np.sum((x[i] - x[k]) ** 2) for i in range(n) for k in range(i + 1, n))
    return x - x[0], L                                                  # put the target at the origin


rng5 = np.random.default_rng(5)
group = ["4v4r"] + [str(rng5.choice(CAST)) for _ in range(6)]           # target ribosome + 6 random neighbours
g_rad = np.array([RAD[p] for p in group])
runs = [compact_pack(g_rad, rng5) for _ in range(5)]
best_x, best_L = min(runs, key=lambda r: r[1])
print("compactness L of 5 random starts:", [f"{L:.0f}" for _, L in runs], "→ keep", f"{best_L:.0f}")
print("neighbour distances from the target (nm):", np.round(np.linalg.norm(best_x[1:], axis=1), 1))

# render a target-centred 64³ subtomogram with the toolkit's imaging shortcut
S5 = 64
scene = np.zeros((S5, S5, S5), np.float32)


def add_particle(vol: np.ndarray, template: np.ndarray, center_zyx, R: np.ndarray) -> None:
    """Add a rotated template into vol (in place), its box centred at center_zyx, clipped at the edges."""
    sub = rotate_volume(template, R)
    lo = np.round(np.asarray(center_zyx)).astype(int) - np.array(sub.shape) // 2
    sl_v, sl_s = [], []
    for a in range(3):
        a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], vol.shape[a])
        if a1 <= a0:
            return
        sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
    vol[tuple(sl_v)] += sub[tuple(sl_s)]


for p, c in zip(group, best_x):
    add_particle(scene, zoo[p], c + (S5 - 1) / 2, random_rotation(rng5))
sub_tomo = simulate_tomogram(scene, np.random.default_rng(6), snr=0.1, tilt_max_deg=60, ctf_pixel_A=10.0, defocus_um=4.0)
fig, axes = plt.subplots(1, 3, figsize=(9, 3.0))
axes[0].imshow(scene.sum(axis=0)); axes[0].set_title("target + neighbours (projection along z)")
axes[1].imshow(scene[S5 // 2]); axes[1].set_title("central xy slice (truth)")
axes[2].imshow(sub_tomo[S5 // 2]); axes[2].set_title("simulated subtomogram, SNR 0.1")
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In about a second we produced one labelled training example: a ribosome at the centre with
neighbours packed against it (some sit above or below it in z, so in the projection they merge with it), imaged with
CTF, missing wedge and noise. Repeat 10,000 times with random neighbours and
orientations and you have a training set for Modules 16–20 — including the realistic difficulty that the box always
contains pieces of *other* molecules.

### 📄 Paper connection — Liu *et al.* 2020 (BIBM): efficient crowding simulation, applied to SARS-CoV-2

**Paper**: Liu S, Ma Y, Ban X, Zeng X, Nallapareddy V, Chaudhari A, Xu M. *Efficient Cryo-Electron Tomogram Simulation of
Macromolecular Crowding with Application to SARS-CoV-2.* 2020 IEEE International Conference on Bioinformatics and
Biomedicine (BIBM), pp. 80–87. [doi:10.1109/BIBM49941.2020.9313185](https://doi.org/10.1109/BIBM49941.2020.9313185)

> ⚠️ We could not access the full text of this paper while writing; the description below is limited to the abstract and
> the AITom code that cites it. Read the paper before relying on details.

- **Question it answers:** how to generate many labelled subtomograms of a target surrounded by realistic crowding
  neighbours, without simulating whole cells.
- **What the paper does (abstract + code):** packs a target and a few randomly chosen neighbours tightly and simulates
  cryo-ET imaging of the small scene; applied to SARS-CoV-2 interacting with a host cell (the simulated images show the
  viral binding domain and the host cell); two classification models trained on the simulated data indicate its
  usefulness. The AITom single-ball code takes normalised gradient steps of $L$ with overlap rejection and keeps the best
  of several runs (this equation is from the code, not verified against the paper).
- **What our toy keeps:** the compaction rule, overlap rejection, best-of-several runs, a target-centred subtomogram.
- **What our toy leaves out:** the SARS-CoV-2 application, the real imaging pipeline of the paper, multi-ball options,
  and the classification experiments. No numbers were verified.
- **Code:** AITom `aitom/geometry/pack/sphere/few/` (`packing_single_sphere/`, `map_tomo/`; the README says it packs only
  5–10 macromolecules so that it runs quickly on CPU).

---
## 6 · From spheres to densities: crowding changes what "SNR" means

A packing gives centres; now we put a **randomly rotated copy of each molecule's density** at each centre (Module 02
rotations). Then we ask two practical questions:

1. If the microscope dose (hence the noise level σ) is fixed, does the tomogram's SNR depend on crowding?
   (Recall the course definition: SNR = var(signal) / var(noise), measured over the whole box.)
2. How much of a ribosome's subtomogram (a 40 nm box around it) is actually *other* molecules?

In [ ]:
# ── 6.1 Render the three packed cells as density maps ──
def render_cell(classes, centres, shape, rng):
    """Place a randomly rotated template at every centre. Returns (volume, list of particle dicts)."""
    vol = np.zeros(shape, np.float32)
    particles = []
    for k, c in zip(classes, centres):
        R = random_rotation(rng)
        add_particle(vol, zoo[k], c, R)
        particles.append(dict(cls=k, center_zyx=np.asarray(c, float), R=R, radius=RAD[k]))
    return vol, particles


t0 = time.time()
cells = {phi: render_cell(compositions[phi][0], sa_results[phi][0], BOX, np.random.default_rng(4)) for phi in TARGETS}
print(f"rendered 3 cells of {BOX} nm in {time.time() - t0:.1f} s")
fig, axes = plt.subplots(1, 3, figsize=(11, 3.7))
for ax, phi in zip(axes, TARGETS):
    vol, parts = cells[phi]
    ax.imshow(vol[BOX[0] // 2]); ax.set_title(f"{100 * phi:.0f} % crowding: {len(parts)} molecules (mid xy slice)")
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.2 Fixed dose: box SNR and neighbour contamination versus crowding ──
SIGMA_FIXED = np.sqrt(cells[0.11][0].var() / 0.05)          # noise level giving SNR 0.05 in the least crowded cell
SUB = 40                                                      # subtomogram box (nm)
snr_box, contamination = [], []
for phi in TARGETS:
    vol, parts = cells[phi]
    snr_box.append(vol.var() / SIGMA_FIXED ** 2)
    fr = []
    for p in parts:
        if p["cls"] != "4v4r":
            continue
        own = np.zeros_like(vol)
        add_particle(own, zoo["4v4r"], p["center_zyx"], p["R"])
        lo = np.maximum(np.round(p["center_zyx"]).astype(int) - SUB // 2, 0)
        sl = tuple(slice(l, l + SUB) for l in lo)
        tot, mine = vol[sl], own[sl]
        fr.append(((tot - mine) ** 2).sum() / (tot ** 2).sum())
    contamination.append(100 * np.mean(fr))
    print(f"{100 * phi:3.0f} % crowding: box SNR at fixed dose = {snr_box[-1]:.3f}; "
          f"neighbours carry {contamination[-1]:4.1f} % of the signal energy in a ribosome's {SUB}³ box ({len(fr)} ribosomes)")
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.0))
axes[0].plot([100 * p for p in TARGETS], snr_box, "o-"); axes[0].set_xlabel("crowding (%)")
axes[0].set_ylabel("SNR = var(signal)/var(noise)"); axes[0].set_title("same dose, different 'SNR'")
axes[1].plot([100 * p for p in TARGETS], contamination, "s-", color="tab:red"); axes[1].set_xlabel("crowding (%)")
axes[1].set_ylabel("neighbour share of box signal (%)"); axes[1].set_title(f"contamination of a {SUB}³ ribosome box")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the *same* noise, the more crowded cell has a higher "SNR" — simply because there is more
signal in the box. A ribosome is not easier to see in a crowded cell, though: it is *harder*, because a growing share of
its subtomogram belongs to neighbours. Two lessons for reading papers:

- ⚠️ **"SNR 0.05" is not a universal quantity.** It depends on the definition (variance ratio over a box, per-particle,
  correlation-based — Module 04), on box size and on crowding. Always check how a paper defines it before comparing
  numbers across papers.
- 🧫 Neighbour contamination is why alignment and classification in crowded cells need target masking or segmentation
  (Module 13, Xu & Alber 2013) and why picking recall drops with crowding (Pei *et al.* 2016; Module 11).

---
## 7 · Full physical simulation versus the fast shortcut

Until now the course has used `simulate_tomogram`, a three-line **fast simulator**: filter the clean volume by a 3D
CTF, zero the missing wedge in Fourier space, add white noise. A **physical** simulator instead mimics the experiment
step by step (as Pei *et al.* did):

1. rotate and place molecules (done: Section 6);
2. record a **tilt series**: −60°…+60° in 2° steps, each image a projection (Module 07);
3. apply the **CTF** of each tilt image (Module 06; here 300 kV, defocus 4 µm underfocus, 10 Å pixels) and the detector
   **MTF** of the form $\mathrm{sinc}(\pi\omega/2)$ (we take $\omega$ as the frequency in units of Nyquist, which gives
   the standard square-pixel MTF), and add shot noise (we use Gaussian noise; Pei *et al.* also add noise before the CTF);
4. reconstruct with **weighted back-projection** (Module 07).

💡 **Why should the shortcut work at all?** By the projection-slice theorem (Module 07), filtering every tilt image by a
CTF that depends only on the in-plane frequency $|k|$ is the same as filtering the 3D volume by a CTF that depends on the
3D $|k|$ — *provided* every tilt has the same defocus. Real tilted specimens violate this a little: defocus changes
across a tilted image (a *defocus gradient* of $x\tan\theta$; for our 80 nm field at 60° only ±0.07 µm, but ±0.9 µm across a
1 µm field) and from tilt to tilt.

We use a small, ~26 %-crowded volume (40 × 80 × 80 nm) so that the 61-image tilt series stays fast. To avoid losing
material at high tilt, the slab is placed in a padded box before projecting (Module 07: content must stay inside the
rotation circle).

In [ ]:
# ── 7.1 A small crowded volume for the physical simulation ──
SHAPE7 = (40, 80, 80)
cls7, rad7 = draw_composition(SHAPE7, 0.26, np.random.default_rng(11))
cen7, _ = sa_pack(rad7, SHAPE7, np.random.default_rng(12))
clean7, parts7 = render_cell(cls7, cen7, SHAPE7, np.random.default_rng(13))
print(f"{len(parts7)} molecules, largest overlap {max_overlap(cen7, rad7):+.2f} nm:",
      {NAMES[p]: cls7.count(p) for p in CAST})

In [ ]:
# ── 7.2 The physical pipeline: tilt series → per-tilt CTF + MTF + noise → WBP ──
def physical_tomogram(clean: np.ndarray, angles_deg, rng, snr: float = 0.05, pixel_A: float = 10.0,
                      defocus_um: float = 4.0, pad: int = 90):
    """Simulate a tomogram the physical way. Returns dict(tomo, signal, noise, per_tilt_snr, tilts_clean, tilts_noisy).

    The pipeline is linear, so signal and noise are reconstructed separately; the per-tilt noise level is then chosen so
    that the final tomogram has var(signal)/var(noise) = snr (to compare with the fast simulator at equal SNR).
    """
    nz, ny, nx = clean.shape
    big = np.zeros((pad, ny, pad), np.float32)                          # slab inside a square (z, x) box
    z0, x0 = (pad - nz) // 2, (pad - nx) // 2
    big[z0:z0 + nz, :, x0:x0 + nx] = clean
    proj = project_tilt_series(big, angles_deg)                         # [n_tilts, y, x]
    ctf = ctf_2d(proj.shape[1:], pixel_A, defocus_um=defocus_um, kV=300)  # same defocus for every tilt (no gradient)
    mtf = np.sinc(radial_frequency(proj.shape[1:]))                     # = sin(πω/2)/(πω/2), ω = |k|/Nyquist

    def filt(imgs, f):
        return np.real(np.fft.ifft2(np.fft.fft2(imgs, axes=(1, 2)) * f, axes=(1, 2))).astype(np.float32)

    sig_tilts = filt(proj, -ctf * mtf)                                  # sign flip as in the toolkit: dense = bright
    noise_tilts = filt(rng.normal(0, 1, proj.shape).astype(np.float32), mtf)  # shot noise, blurred by the detector
    crop = (slice(z0, z0 + nz), slice(None), slice(x0, x0 + nx))
    rec_s = weighted_back_projection(sig_tilts, angles_deg, pad)[crop]
    rec_n = weighted_back_projection(noise_tilts, angles_deg, pad)[crop]
    sigma = np.sqrt(rec_s.var() / (snr * rec_n.var()))
    return dict(tomo=rec_s + sigma * rec_n, signal=rec_s, noise=sigma * rec_n,
                per_tilt_snr=float(sig_tilts.var() / (sigma ** 2 * noise_tilts.var())),
                tilts_clean=proj, tilts_noisy=sig_tilts + sigma * noise_tilts)


ANG = np.arange(-60, 61, 2.0)                                           # Pei et al.'s tilt scheme: 61 images
SNR7 = 0.05
t0 = time.time()
phys = physical_tomogram(clean7, ANG, np.random.default_rng(14), snr=SNR7)
t_phys = time.time() - t0
t0 = time.time()
fast = simulate_tomogram(clean7, np.random.default_rng(15), snr=SNR7, tilt_max_deg=60, ctf_pixel_A=10.0, defocus_um=4.0)
fast_signal = apply_fourier_mask(apply_fourier_mask(clean7, -ctf_2d(SHAPE7, 10.0, defocus_um=4.0, kV=300)),
                                 missing_wedge_mask(SHAPE7, 60))
fast_noise = fast - fast_signal
t_fast = time.time() - t0
print(f"physical simulation: {len(ANG)} tilts, {t_phys:.1f} s | fast simulation: {t_fast:.2f} s")
print(f"per-tilt image SNR in the physical simulation ≈ {phys['per_tilt_snr']:.3f} (final tomogram SNR {SNR7})")
print(f"noise-free parts:  corr(physical, fast) = {pearson(phys['signal'], fast_signal):.3f}")
print(f"noisy tomograms:   corr with truth — physical {pearson(phys['tomo'], clean7):.3f}, fast {pearson(fast, clean7):.3f}")
assert pearson(phys["signal"], fast_signal) > 0.9, "the CTF + wedge shortcut should match the noise-free physics"

In [ ]:
# ── 7.3 What the virtual microscope recorded: three tilt images ──
fig, axes = plt.subplots(2, 3, figsize=(9, 5.4))
for col, a in enumerate([0, 30, 60]):
    j = int(np.where(ANG == a)[0][0])
    axes[0, col].imshow(phys["tilts_clean"][j]); axes[0, col].set_title(f"{a}°: projection (no CTF, no noise)", fontsize=8)
    axes[1, col].imshow(phys["tilts_noisy"][j]); axes[1, col].set_title(f"{a}°: recorded image (CTF + noise)", fontsize=8)
    for r in range(2):
        axes[r, col].set_xticks([]); axes[r, col].set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# ── 7.4 Physical vs fast tomograms, side by side ──
# The pipeline is linear, so a tomogram at any SNR is signal + scaled noise. At the realistic SNR 0.05 both look like
# the noise of 7.3; we show the noise-free parts and SNR 0.5 (10× less noise) so the textures can be compared.
SNR_SHOW = 0.5
k_noise = np.sqrt(SNR7 / SNR_SHOW)
panels = [(clean7, "truth"), (phys["signal"], "physical, no noise"), (fast_signal, "fast, no noise"),
          (phys["signal"] + k_noise * phys["noise"], f"physical, SNR {SNR_SHOW}"),
          (fast_signal + k_noise * fast_noise, f"fast, SNR {SNR_SHOW}")]
zc, yc = SHAPE7[0] // 2, SHAPE7[1] // 2
fig, axes = plt.subplots(2, 5, figsize=(13, 4.6), gridspec_kw=dict(height_ratios=[2, 1.1]))
for col, (vol, t) in enumerate(panels):
    axes[0, col].imshow(vol[zc]); axes[0, col].set_title(f"{t}\nxy slice", fontsize=8)
    axes[1, col].imshow(vol[:, yc, :]); axes[1, col].set_title("xz slice", fontsize=8)
    for r in range(2):
        axes[r, col].set_xticks([]); axes[r, col].set_yticks([])
axes[0, 0].set_ylabel("y"); axes[1, 0].set_ylabel("z")
plt.tight_layout(); plt.show()

In [ ]:
# ── 7.5 Where do the two simulators differ? The noise ──
def radial_power(vol: np.ndarray, n_bins: int = 20):
    """Spherically averaged power spectrum: (frequency in cycles/nm, mean |F|² per shell)."""
    k = radial_frequency(vol.shape)
    idx = np.minimum((k / 0.5 * n_bins).astype(int), n_bins)
    p = np.bincount(idx.ravel(), (np.abs(np.fft.fftn(vol)) ** 2).ravel(), n_bins + 1)
    c = np.bincount(idx.ravel(), minlength=n_bins + 1)
    return (np.arange(n_bins) + 0.5) / n_bins * 0.5, (p / np.maximum(c, 1))[:n_bins]


fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for vol, lab in [(phys["noise"], "physical: noise"), (fast_noise, "fast: noise")]:
    f, pw = radial_power(vol)
    axes[0].semilogy(f, pw / pw.mean(), "o-", ms=3, label=lab)
axes[0].set_xlabel("spatial frequency (1/nm)"); axes[0].set_ylabel("noise power (normalised)")
axes[0].set_title("noise power spectrum"); axes[0].legend(fontsize=7)
for ax, vol, t in [(axes[1], phys["noise"], "physical noise"), (axes[2], fast_noise, "fast noise")]:
    spec = np.log1p(np.abs(np.fft.fftshift(np.fft.fftn(vol))) ** 2).mean(axis=1)   # (k_z, k_x), mean over k_y
    ax.imshow(spec, origin="lower", extent=(-0.5, 0.5, -0.5, 0.5), aspect="auto", cmap="viridis",
              vmin=np.percentile(spec, 2), vmax=np.percentile(spec, 99.8))
    ax.set_title(f"{t}: log power in ($k_z$, $k_x$)"); ax.set_xlabel("$k_x$ (1/nm)"); ax.set_ylabel("$k_z$ (1/nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?**

- **Noise-free, the two simulators agree** (correlation printed in 7.2 well above 0.9; compare the second and third
  columns of 7.4): CTF + missing wedge is an excellent model of the physical pipeline when defocus is the same for every
  tilt. That is why the fast simulator is fine for most exercises in this course.
- At SNR 0.5 you can already see the different **noise textures**: fine-grained white speckle (fast) versus streaky,
  direction-dependent noise (physical).
- **The noise is where they differ.** In the physical simulation the noise was added to each tilt image and then went
  through the ramp filter of WBP: its power **rises with frequency** and it has **its own missing wedge** (dark wedge in
  the middle panel). The fast simulator adds **white** noise *after* the wedge mask, so its noise is flat and fills the
  wedge region too (right panel). Physical tomograms also carry streak artefacts from the discrete tilts. (The bright
  line at $k_z=0$ in the middle panel is an interpolation artefact: the 0° image is the only one back-projected without
  interpolation, so its plane keeps more high-frequency noise.)
- Remember 7.3: each recorded tilt image is almost pure noise (per-tilt SNR printed in 7.2), yet 61 of them combine into
  a usable tomogram.

| | Fast `simulate_tomogram` | Physical simulation |
|---|---|---|
| CTF | one 3D filter, isotropic | per tilt image (equivalent if defocus is constant) |
| Missing wedge | Fourier mask | emerges from the tilt geometry |
| Noise | white, also inside the wedge | added per tilt; ramp-weighted (rises with frequency), wedge-shaped |
| Artefacts | none | streaks from discrete tilts, interpolation |
| Cost here | ≈ 0.1 s | many seconds (61 projections + 2 back-projections) |

⚠️ A network trained on fast-simulated noise has never seen ramp-coloured, wedge-shaped noise — a small **domain shift**
even between two simulators (Module 20).

✏️ **Your turn.** In 7.2 change `defocus_um=4.0` to `2.0` in *both* simulators (and re-run 7.2–7.5). How do the slices
change? (Hint: Module 06 — where is the first CTF zero?)

---
## 8 · The answer key: a ground-truth table, and community benchmarks

The whole point of simulation is the answer key. We save one row per molecule — class, centre, orientation — in a CSV
file, the same kind of annotation that benchmark datasets ship with. Orientations are stored as **ZYZ Euler angles**
(Module 02), and we check that converting back gives the same rotation.

In [ ]:
# ── 8.1 Save the ground truth of the 7.x volume ──
GT_FILE = "in_silico_cell_ground_truth.csv"
with open(GT_FILE, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "pdb", "name", "z_nm", "y_nm", "x_nm", "phi_deg", "theta_deg", "psi_deg", "bounding_radius_nm"])
    for i, p in enumerate(parts7):
        phi, theta, psi = matrix_to_euler_zyz(p["R"])
        assert np.allclose(euler_zyz_to_matrix(phi, theta, psi), p["R"], atol=1e-6), "Euler round trip failed"
        w.writerow([i, p["cls"].upper(), NAMES[p["cls"]], *np.round(p["center_zyx"], 2),
                    *np.round(np.degrees([phi, theta, psi]), 2), round(p["radius"], 2)])
print(f"wrote {len(parts7)} rows to {GT_FILE}\n")
with open(GT_FILE) as f:
    for line in list(f)[:6]:
        print(line.rstrip())

🔍 **What just happened?** Every molecule in the simulated tomogram is now documented. A picker (Module 11) is scored by
matching its detections to the `z, y, x` columns; a classifier (Module 16) by the `pdb` column; an aligner (Modules 12,
19) by the Euler angles. This is precisely what a community benchmark provides.

### 📄 Paper connection — Gubins *et al.*: SHREC'19 and SHREC'20 classification in cryo-electron tomograms

**Papers**: Gubins I, van der Schot G, Veltkamp RC, Förster F, Du X, Zeng X, Zhu Z, Chang L, Xu M, *et al.*
*Classification in Cryo-Electron Tomograms* (SHREC'19 track). Eurographics Workshop on 3D Object Retrieval (2019).
[doi:10.2312/3dor.20191061](https://doi.org/10.2312/3dor.20191061) · Gubins I, Chaillet ML, van der Schot G, Veltkamp RC,
Förster F, Hao Y, Wan X, Cui X, Zhang F, Moebel E, Wang X, Kihara D, Zeng X, Xu M, Nguyen NP, White T, Bunyak F.
*SHREC 2020: Classification in cryo-electron tomograms.* Computers & Graphics 91:279–289 (2020).
[doi:10.1016/j.cag.2020.07.010](https://doi.org/10.1016/j.cag.2020.07.010)

- **Question they answer:** how well do different methods *localise and classify* proteins in tomograms, judged on a
  public, fully annotated **simulated** dataset that everyone uses in the same way?
- **Benchmark design:** SHREC'19 — 10 simulated cell-like tomograms with 12 protein types (9 for training, 1 for testing);
  SHREC'20 — 12 protein classes plus vesicles and gold fiducials, molecular weights from 42.75 kDa (PDB 1S3X — the
  "too small to see" Hsp70 domain of our zoo) to 3,325.59 kDa (5MRC — also in our zoo), 1,000–3,000 proteins per
  volume, 9 tomograms (0–7 train/validation, 8 test), 61 tilts over ±60°, 300 kV, defocus 2–5 µm, dose 100–120 e⁻/Å²,
  SNR 0.12–0.58 (SHREC'20 appeared in *Computers & Graphics*, not in the 3DOR proceedings).
- **Metrics:** precision, recall, $F_1=2PR/(P+R)$, miss rate, average centre distance, multiple hits.
- **Result reported:** in SHREC'20, every learning-based method beat template matching ($F_1$ 0.372 for TM, 0.576 for its
  variant TM-F); DeepFinder had the best overall $F_1$ (0.868); performance correlated strongly with molecular weight
  (e.g. MC DS Net: $F_1$ 0.594 below 200 kDa vs 0.989 above 600 kDa). The Xu Lab participated (Zeng X and Xu M are
  co-authors); which entry was the lab's is not verified here.
- **Connection to this module:** SHREC data are exactly the product of Sections 4–8 at scale. Module 16 trains 3D CNNs
  on such data and discusses evaluation pitfalls (split by tomogram!).

---
## 9 · Pseudo-structures, and the limits of simulation

### 9.1 Molecules that do not exist — on purpose

Template matching (Module 11) always returns *some* best match, even on pure noise. To decide whether a match is
**statistically significant**, Wang *et al.* (2018) compare the score of the best template with scores obtained from many
**pseudo-structures**: realistic-looking, protein-like density maps that are *not* the molecule in question. If a real
template does not beat almost all pseudo-templates, the match is not credible. They learned to generate such shapes with
a 3D generative adversarial network; here we only illustrate the idea with a very cheap generator — random clusters of
Gaussian blobs of protein-like size.

In [ ]:
# ── 9.1 A cheap pseudo-structure generator (NOT the paper's GAN) ──
def pseudo_structure(rng, size: int = 32, n_blobs=(4, 10), spread: float = 5.0) -> np.ndarray:
    """A protein-like random density: a few Gaussian blobs around the centre, blurred, scaled to [0, 1]."""
    g = np.arange(size) - (size - 1) / 2
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    vol = np.zeros((size,) * 3, np.float32)
    for _ in range(rng.integers(*n_blobs)):
        c = rng.normal(0, spread, 3)
        s = rng.uniform(1.8, 3.5)
        vol += np.exp(-((z - c[0]) ** 2 + (y - c[1]) ** 2 + (x - c[2]) ** 2) / (2 * s * s))
    vol = ndimage.gaussian_filter(vol, 1.0)
    return vol / vol.max()


rng9 = np.random.default_rng(9)
pseudo = [pseudo_structure(rng9) for _ in range(6)]
fig, axes = plt.subplots(1, 8, figsize=(12, 1.9))
for ax, (v, t) in zip(axes, [(zoo["1bxn"], "RuBisCO (real)"), (zoo["3j9i"], "20S (real)")] +
                      [(p, f"pseudo #{i + 1}") for i, p in enumerate(pseudo)]):
    ax.imshow(v.sum(axis=0)); ax.set_title(t, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
print("correlation of each pseudo-structure with RuBisCO (no alignment):",
      [f"{pearson(p, zoo['1bxn']):.2f}" for p in pseudo])

🔍 **What just happened?** The pseudo-structures have protein-like size and lumpiness but no biological identity — a
"null population" of shapes. Module 11 uses them to turn a template-matching score into a p-value.

### 📄 Paper connection — Wang *et al.* 2018: pseudo-structures for statistical template matching

**Paper**: Wang KW, Zeng X, Liang X, Huo Z, Xing EP, Xu M. *Image-derived generative modeling of pseudo-macromolecular
structures – towards the statistical assessment of Electron CryoTomography template matching.* British Machine Vision
Conference (BMVC) 2018 (venue per the lab bibliography); [arXiv:1805.04634](https://arxiv.org/abs/1805.04634).

- **Question it answers:** a cross-correlation score says which template fits *best*, not whether the fit is credible —
  how can we test the hypothesis $H_0: P\neq T(C_I)$ (the subtomogram $P$ is not the top-scoring complex $C_I$) against
  $H_A: P=T(C_I)$ when there are too few known structures to build a null distribution?
- **What the paper actually does:** trains a 3D Wasserstein GAN with gradient penalty (λ = 10; 3D-DCGAN-style networks,
  100-dimensional latent vector) on rotated copies of 15 PDB-derived maps (64³ at 0.6 nm); samples ≈ 1,000
  pseudo-structures "far" from $C_I$ (rejection region built from an inverse generator, a kernel density estimate and a
  Bayes classifier); computes the Monte-Carlo p-value
  $$\hat p=\frac1B\sum_{b=1}^{B}\mathbb 1\big[c(P,T(C_I))\le c(P,T(C_0^{(b)}))\big],$$
  *in words:* the fraction of pseudo-templates that match the subtomogram at least as well as the best real template;
  a match is claimed if $\hat p\le 0.01$.
- **What our toy keeps:** the idea of a population of realistic non-matching shapes.
- **What our toy leaves out:** the GAN, the rejection region, the test itself (Module 11 builds it).
- **Result reported:** the authors report 12 of 15 simulated tests succeeding; 264 of 376 (70.2 %) experimental ribosome
  subtomograms from EMPIAR-10045 passing (92 of the top 100 by alignment score); and that the test prevents 40 % of the
  false positives of the highest-score rule on 20 empty subtomograms.
- **Code:** none located in AITom.

### 9.2 What our simulator (and most simulators) leave out

| Real tomograms have… | Our simulator | Consequence |
|---|---|---|
| hundreds of molecular species, many unknown | 4 species | classifiers meet "unknown" molecules (Modules 15, 19, 22) |
| non-Gaussian, structured noise; ice thickness gradients | white/ramp Gaussian noise | denoisers and classifiers trained on simulation degrade |
| beam-induced motion, radiation damage, alignment errors | none | blurrier, less consistent data |
| defocus gradients and per-tilt defocus changes | one defocus | CTF artefacts differ |
| membranes with lipids and embedded proteins, filaments, nucleic acids | one vesicle (Module 00) or bead chains | segmentation trained on toys fails on real membranes |
| crowding by molecules smaller than our voxels | none | background texture differs |

The gap between simulated and real data is called **domain shift** (Module 20). The lab attacks it from both sides:
**making simulations more realistic by learning** — CryoETGAN (Wu *et al.* 2022, Front Physiol, doi:10.3389/fphys.2022.760404)
translates clean density maps into realistic-looking subtomograms with an unpaired 3D CycleGAN, and DUAL (Zeng *et al.*,
bioRxiv 2024, doi:10.1101/2024.03.02.583135) learns simulation and denoising simultaneously from unpaired data — and
**making models robust** to the shift (domain adaptation, Module 20).

---
## ✅ Checkpoint

1. Why can't we benchmark a particle picker on a real cellular tomogram alone?
   <details><summary>Answer</summary>Because the true identities, positions and orientations of the molecules are unknown; any "ground truth" is an expert's guess. Simulation provides an exact answer key (at the cost of realism).</details>
2. Two spheres of radius 15 nm and 8 nm have centres 20 nm apart. Do they overlap, and what is their overlap-energy term?
   <details><summary>Answer</summary>Yes: r₁ + r₂ − d = 23 − 20 = 3 nm > 0, so the pair contributes 3² = 9 nm² to E.</details>
3. In simulated annealing at temperature T = 2 nm², a move raises the energy by 2 nm². With what probability is it accepted? And at T = 0.02 nm²?
   <details><summary>Answer</summary>e^(−2/2) = e^(−1) ≈ 0.37; at T = 0.02, e^(−100) ≈ 0 — essentially never.</details>
4. Why does rejection sampling fail at 44 % crowding while annealing succeeds?
   <details><summary>Answer</summary>Rejection sampling never moves placed molecules; random placements leave gaps too small for later (especially large) molecules and the box jams (≈ 38 % for equal spheres in 3D). Annealing starts with overlaps and lets all molecules move, so it can reorganise the packing to remove them.</details>
5. The fast and the physical simulators produce almost identical noise-free tomograms. What is the main remaining difference?
   <details><summary>Answer</summary>The noise: physical noise is added per tilt image and ramp-filtered by WBP (rising with frequency, with its own missing wedge, plus streaks), whereas the fast simulator adds white noise after the wedge mask (flat, fills the wedge).</details>
6. Two papers both report results "at SNR 0.05". Name two reasons the numbers might not be comparable.
   <details><summary>Answer</summary>Different SNR definitions (variance ratio vs correlation-based vs per-tilt), different box sizes or crowding (more signal in the box raises the variance-ratio SNR at the same dose), different simulators (white vs ramp-coloured noise, CTF, wedge).</details>

## 🧠 Questions & discussion

1. Pei *et al.* found that picking recall falls as crowding rises. Using Section 6, explain two mechanisms behind this.
2. Our annealing used Monte-Carlo moves; Liu *et al.* used molecular dynamics. What does each approach need to know
   (energies, forces, masses), and which would you choose for 10⁵ molecules?
3. Representing a molecule by bounding spheres guarantees no overlap but wastes space. What could go wrong biologically
   if your simulator packs molecules *less* tightly than a real cell does?
4. Should a network trained on SHREC data be expected to work on a real *Mycoplasma* tomogram? What would you check first?
5. A pseudo-structure generator that is *too* similar to real proteins, or too different, would bias Wang *et al.*'s
   test. In which direction would each mistake push the p-values?

## 🚀 Bonus — going further

- **More species:** add `"1qvr"` (ClpB) and `"3cf3"` (p97) to `CAST` and `MIX` (re-run from Section 0) and repeat Section 4.
  How does the composition of a cell change the reachable crowding?
- **Phase flipping:** in `physical_tomogram`, multiply each noisy tilt image's spectrum by `sign(ctf)` (CTF correction by
  phase flipping) before back-projection. Compare the xy slices with 7.4.
- **Defocus gradient:** give each tilt a different defocus (e.g. `defocus_um=4 + 0.3 * rng.normal()`) and measure how the
  noise-free agreement with the fast simulator in 7.2 drops.
- **Real code:** explore AITom's simulation modules (`aitom/simulation/tomogram/single_bounding_sphere/`,
  `aitom/geometry/pack/sphere/few/`) on <https://github.com/xulabs/aitom>, and the PolNet simulator
  (<https://github.com/anmartinezs/polnet>), a full-featured tomogram simulator; read how a SHREC dataset's annotation
  file is organised.
- **Deep dive:** [`../../SubtomogramAveraging/`](../../SubtomogramAveraging/) simulates a multi-particle tomogram and
  averages it.

## 📝 Lab note

```text
Module 09 — date:
Input: code-generated zoo maps (4V4R, 3J9I, 6UTJ, 1BXN); target crowding 11/26/44 %
Output: packings (rejection vs annealing), rendered cells, physical and fast tomograms, ground-truth CSV
Method: bounding spheres, Metropolis simulated annealing, multi-ball k-means, bead-spring membrane, gradient compaction,
        tilt series + CTF + MTF + noise + WBP
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Pei et al. 2016; Liu et al. 2020 (BMC Bioinf); Liu et al. 2020 (BIBM); SHREC'19/'20; Wang et al. 2018
```

## 📚 References

1. Pei L, Xu M, Frazier Z, Alber F. Simulating cryo electron tomograms of crowded cell cytoplasm for assessment of automated particle picking. *BMC Bioinformatics* 17:405 (2016). doi:10.1186/s12859-016-1283-3
2. Liu S, Ban X, Zeng X, Zhao F, Gao Y, Wu W, Zhang H, Chen F, Hall T, Gao X, Xu M. A unified framework for packing deformable and non-deformable subcellular structures in crowded cryo-electron tomogram simulation. *BMC Bioinformatics* 21:399 (2020). doi:10.1186/s12859-020-03660-w
3. Liu S, Ma Y, Ban X, Zeng X, Nallapareddy V, Chaudhari A, Xu M. Efficient cryo-electron tomogram simulation of macromolecular crowding with application to SARS-CoV-2. *IEEE BIBM* 2020, pp. 80–87. doi:10.1109/BIBM49941.2020.9313185
4. Gubins I *et al.* Classification in cryo-electron tomograms (SHREC'19). *Eurographics Workshop on 3D Object Retrieval* (2019). doi:10.2312/3dor.20191061
5. Gubins I *et al.* SHREC 2020: Classification in cryo-electron tomograms. *Computers & Graphics* 91:279–289 (2020). doi:10.1016/j.cag.2020.07.010
6. Wang KW, Zeng X, Liang X, Huo Z, Xing EP, Xu M. Image-derived generative modeling of pseudo-macromolecular structures – towards the statistical assessment of electron cryotomography template matching. *BMVC* (2018). arXiv:1805.04634
7. Wu X, Li C, Zeng X, Wei H, Deng HW, Zhang J, Xu M. CryoETGAN: cryo-electron tomography image synthesis via unpaired image translation. *Front Physiol* 13:760404 (2022). doi:10.3389/fphys.2022.760404
8. Zeng X, Ding Y, Zhang Y, Uddin M, Dabouei A, Xu M. DUAL: deep unsupervised simultaneous simulation and denoising for cryo-electron tomography. *bioRxiv* (2024). doi:10.1101/2024.03.02.583135
9. Kirkpatrick S, Gelatt CD, Vecchi MP. Optimization by simulated annealing. *Science* 220:671–680 (1983). doi:10.1126/science.220.4598.671
10. Metropolis N, Rosenbluth AW, Rosenbluth MN, Teller AH, Teller E. Equation of state calculations by fast computing machines. *J Chem Phys* 21:1087–1092 (1953). doi:10.1063/1.1699114
11. Zimmerman SB, Trach SO. Estimation of macromolecule concentrations and excluded volume effects for the cytoplasm of *Escherichia coli*. *J Mol Biol* 222:599–620 (1991). doi:10.1016/0022-2836(91)90499-V
12. Martinez-Sanchez A *et al.* PolNet — a full simulator of cryo-ET tomograms (for comparison with our toy). <https://github.com/anmartinezs/polnet>

## ➡️ Next

**[10 · How good is a map? FSC, half-maps, local resolution and particle selection](10_resolution_validation.ipynb)** —
now that we can make data with a known answer, we learn how to measure quality when the answer is *not* known.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")